# 02 - Train and verify the official Sleep DSS model

This notebook runs the same `src/train.py` workflow as `python src/train.py`. The official `models/model.joblib` artifact is one `sklearn Pipeline` containing preprocessing and the classifier; this notebook does not train a separate Logistic Regression model.

After training, the notebook checks `predict_proba` and compares its risk probability with the `predict_risk` function used by the app. The values must match.

Only behavioral and context features are used by the official model. Sleep outcome columns are excluded to prevent data leakage.

In [1]:
from pathlib import Path
import sys

import numpy as np
from sklearn.pipeline import Pipeline

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "train.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not find the project root containing src/train.py.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.preprocess import prepare_data
from src.train import train_and_evaluate
from src.decision import load_model, predict_risk

MODELS = ROOT / "models"
test_metrics, cv_metrics = train_and_evaluate()

model = load_model()
assert isinstance(model, Pipeline), "The official model must be one Pipeline."
assert list(model.named_steps) == ["prep", "model"]
assert model.classes_.tolist() == [0, 1]
print(f"Saved and reloaded model: {MODELS / 'model.joblib'}")
print(f"Pipeline steps: {list(model.named_steps)}")
print(f"Test metrics rows: {len(test_metrics)}; CV rows: {len(cv_metrics)}")

Logistic Regression best params: {'model__C': 1.0, 'model__class_weight': 'balanced'}


Random Forest best params: {'model__class_weight': None, 'model__max_depth': 12, 'model__min_samples_leaf': 3, 'model__n_estimators': 400}


XGBoost best params: {'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 200}


Background dataset has 6800 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=6800 when initializing the masker.


Training rows: 6800; test rows: 1700
Selected model by mean 5-fold CV ROC-AUC: Logistic Regression

5-fold CV mean and standard deviation:
              model  accuracy_mean  accuracy_std  precision_mean  precision_std  recall_mean  recall_std  f1_mean  f1_std  roc_auc_mean  roc_auc_std
      Most frequent         0.6010        0.0004          0.6010         0.0004       1.0000      0.0000   0.7508  0.0003        0.5000       0.0000
Logistic Regression         0.8628        0.0043          0.9093         0.0063       0.8574      0.0079   0.8825  0.0039        0.9408       0.0060
      Random Forest         0.8601        0.0074          0.8915         0.0081       0.8737      0.0076   0.8825  0.0062        0.9356       0.0055
            XGBoost         0.8676        0.0058          0.8960         0.0084       0.8823      0.0081   0.8891  0.0047        0.9407       0.0056

Held-out test metrics:
              model  accuracy  precision  recall     f1  roc_auc
      Most frequent    0.60

In [2]:
_, X_test, _, _ = prepare_data()
sample = X_test.iloc[[0]]
profile = sample.iloc[0].to_dict()

positive_class_index = list(model.classes_).index(1)
pipeline_risk = float(model.predict_proba(sample)[0, positive_class_index])
app_risk = predict_risk(profile)

assert np.isclose(pipeline_risk, app_risk, rtol=0, atol=1e-12), (
    f"Pipeline risk ({pipeline_risk}) does not match app risk ({app_risk})."
)
assert 0.0 <= app_risk <= 1.0
print(f"Risk from Pipeline: {pipeline_risk * 100:.1f}%")
print(f"Risk from app predict_risk: {app_risk * 100:.1f}%")
print("Verified: both risk values match.")

Risk from Pipeline: 21.2%
Risk from app predict_risk: 21.2%
Verified: both risk values match.
